# Player Detection using YOLOv11 (AFL)

### Mounting the Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

### Install Ultralytics

Installing Ultralytics to run the YOLO

In [ ]:
!pip install ultralytics
print("Ultralytics installed!")

### Loading the Model

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv11 nano model
model = YOLO('yolo11n.pt')

### Training the model using the annotated dataset

### creating the data.yml file

In [ ]:
import os

# Create data.yaml directly in the correct Drive folder
yaml_content = """path: /content/drive/MyDrive/Colab Notebooks/Project_Orion/Labelled_Data/yolo_train_data
train: images
val: images

nc: 3
names:
  0: CAR
  1: GCS
  2: REF
"""

yaml_path = '/content/drive/MyDrive/Colab Notebooks/Project_Orion/Labelled_Data/yolo_train_data/data.yaml'

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("data.yaml created successfully at:")
print(yaml_path)

In [ ]:
# Train on your AFL annotated dataset
model.train(
    data='/content/drive/MyDrive/Colab Notebooks/Project_Orion/Labelled_Data/yolo_train_data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='afl_player_detection'
)

* We have the manually labelled dataset to train the model.
* Used Label-studio to manually label the dataset (200 images each)

### Saving the best.pt to Drive

In [ ]:
import os

for root, dirs, files in os.walk('/content/runs'):
    for file in files:
        if file == 'best.pt':
            print(os.path.join(root, file))

In [ ]:
import shutil
import os

os.makedirs('/content/drive/MyDrive/Colab Notebooks/Project_Orion/AFL_Model', exist_ok=True)

shutil.copy(
    '/content/runs/detect/afl_player_detection2/weights/best.pt',
    '/content/drive/MyDrive/Colab Notebooks/Project_Orion/AFL_Model/best.pt'
)
print("New AFL model saved to Drive!")

### Load the new AFL model for inference test:

In [ ]:
from ultralytics import YOLO
from IPython.display import Image as IPImage
import glob

# Load YOUR new AFL trained model
model = YOLO('/content/drive/MyDrive/Colab Notebooks/Project_Orion/AFL_Model/best.pt')

# Test on one image from your training data
results = model.predict(
    source='/content/drive/MyDrive/Colab Notebooks/Project_Orion/Labelled_Data/yolo_train_data/images',
    conf=0.3,
    save=True,
    max_det=50
)

# Show one result
output_images = glob.glob("/content/runs/detect/predict*/*.jpg")
IPImage(output_images[0])

### Loading the model

In [ ]:
from ultralytics import YOLO

# Load your AFL trained model
model = YOLO('/content/drive/MyDrive/Colab Notebooks/Project_Orion/AFL_Model/best.pt')
print("Model loaded successfully!")

### Upload multiple images

In [ ]:
from google.colab import files

print("Upload 6-7 AFL frame images...")
uploaded = files.upload()
image_filenames = list(uploaded.keys())
print(f"\nUploaded {len(image_filenames)} images:")
for img in image_filenames:
    print(f"  → {img}")

###  inference on all images

In [ ]:
from IPython.display import Image as IPImage, display
import glob, os

# Summary table
summary = []

for image_file in image_filenames:
    print(f"\n{'='*40}")
    print(f"Image: {image_file}")
    print(f"{'='*40}")

    results = model.predict(
        source=image_file,
        conf=0.3,
        save=True,
        verbose=False
    )

    boxes = results[0].boxes
    num_detections = len(boxes)

    # Count each class detected
    car_count = 0
    gcs_count = 0
    ref_count = 0

    for box in boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        class_name = model.names[cls]
        print(f"  → {class_name}: {conf:.2f} confidence")

        if class_name == 'CAR':
            car_count += 1
        elif class_name == 'GCS':
            gcs_count += 1
        elif class_name == 'REF':
            ref_count += 1

    print(f"\nSummary for this image:")
    print(f"  Carlton players (CAR): {car_count}")
    print(f"  Gold Coast players (GCS): {gcs_count}")
    print(f"  Referees (REF): {ref_count}")
    print(f"  Total detections: {num_detections}")

    summary.append({
        'image': image_file,
        'CAR': car_count,
        'GCS': gcs_count,
        'REF': ref_count,
        'total': num_detections
    })

# Show all result images
print("\n\n--- DISPLAYING ALL RESULTS ---")
output_images = sorted(glob.glob("/content/runs/detect/predict*/*.jpg"))
for img_path in output_images[-len(image_filenames):]:
    print(f"\n{os.path.basename(img_path)}")
    display(IPImage(img_path, width=800))

### Model confidence metrics

In [ ]:
print("\n=== MODEL CONFIDENCE METRICS ===\n")

all_confs = {'CAR': [], 'GCS': [], 'REF': []}

for image_file in image_filenames:
    results = model.predict(
        source=image_file,
        conf=0.3,
        verbose=False
    )
    for box in results[0].boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        class_name = model.names[cls]
        all_confs[class_name].append(conf)

for class_name, confs in all_confs.items():
    if confs:
        print(f"{class_name}:")
        print(f"  Average confidence: {sum(confs)/len(confs):.2f}")
        print(f"  Highest confidence: {max(confs):.2f}")
        print(f"  Lowest confidence:  {min(confs):.2f}")
        print(f"  Total detections:   {len(confs)}")
    else:
        print(f"{class_name}: No detections found")
    print()
